# Briefcase AI — Pattern Library Walkthrough

A single-notebook tour of the 11 SDK primitives. Each section uses the
corresponding `patterns/NN_*.py` file as its source of truth — this notebook
collapses them for easy reading; the per-pattern notebooks remain available
for isolated study.

**How to use this:** run top-to-bottom for the full tour (~30 seconds), or
jump to the section you care about. Section headings mirror the filenames.

**Install first:** `pip install -r patterns/requirements.txt`


## 1. Decision capture

`@capture` auto-records function invocations as `DecisionSnapshot`s. This is the audit atom — what the system DID, immutably, on every call.

In [ ]:
import briefcase
from briefcase.decorators import capture

# Route captured records to an in-memory exporter so we can inspect them
# ("console" prints to stderr; a "*.jsonl" path appends to a file).
recorder = briefcase.observe("memory")


# async_capture=False records synchronously so we can inspect immediately.
@capture(decision_type="sentiment_classification", context_version="v1", async_capture=False)
def classify(document: str, model: str = "gpt-4o") -> dict:
    """Dummy 'LLM' — deterministic so the pattern runs offline."""
    score = 0.92 if "great" in document.lower() else 0.31
    return {"label": "positive" if score > 0.5 else "negative", "score": score}

# --- Invoke the instrumented function ---
# Every @capture'd call produces a decision record: inputs, outputs,
# started_at, ended_at, execution_time_ms, context_version, and any error
# raised — each keyed by a fresh decision_id (UUID).
result = classify("The product is great", model="gpt-4o")
print(f"Classifier returned: {result}")

# --- A second call produces a separate record ---
result2 = classify("This is terrible", model="gpt-4o")
print(f"Second call:         {result2}")

# --- Inspect the captured audit atoms ---
print(f"\nCaptured {len(recorder.records)} decision record(s):")
for rec in recorder.records:
    print(f"  - {rec['decision_type']} ({rec['execution_time_ms']:.3f} ms): "
          f"inputs={rec['inputs']} -> outputs={rec['outputs']}")

# --- Errors are captured too ---
# If the wrapped function raises, @capture records the exception on the
# record and re-raises it. The audit trail retains failed calls with full
# context — important for debugging and post-mortem analysis.
@capture(decision_type="always_fails", async_capture=False)
def always_fails(x: int) -> int:
    raise ValueError(f"no: {x}")

try:
    always_fails(42)
except ValueError as e:
    print(f"\nCaptured error: {type(e).__name__}: {e}")
    print(f"Total records (including the failed call): {len(recorder.records)}")

## 2. Bitemporal evidence

The capture layer records actions. The bitemporal layer records what was *known* at each action. Every record carries a `valid_time` (world-time) and `transaction_time` (system-time), and the store is append-only by construction.

In [ ]:
from datetime import datetime, timezone

from briefcase.bitemporal import BitemporalRecord, InMemoryBitemporalStore

UTC = timezone.utc

# --- Construct a BitemporalRecord directly ---
# Every record carries key, value, valid_time, transaction_time, source,
# trust level, metadata, and (later) an optional parent_record_id for
# corrections. `BitemporalRecord.new()` assigns a fresh record_id.
record = BitemporalRecord.new(
    key="fx:USDC/USD",
    valid_time=datetime(2026, 4, 17, tzinfo=UTC),
    value={"px": 1.0001, "size": 1_000_000},
    source="bloomberg",
    source_trust_level="primary",
    transaction_time=datetime(2026, 4, 17, tzinfo=UTC),
    metadata={"feed": "BVAL"},
)
print("BitemporalRecord.new(...) produces:")
print(f"  record_id:         {record.record_id}")
print(f"  key:               {record.key}")
print(f"  valid_time:        {record.valid_time.isoformat()}")
print(f"  transaction_time:  {record.transaction_time.isoformat()}")
print(f"  value:             {record.value}")
print(f"  source:            {record.source} (trust={record.source_trust_level})")
print(f"  parent_record_id:  {record.parent_record_id}  (None = original)")

# --- Append to an InMemoryBitemporalStore ---
# The store exposes append(), history(), latest(), as_of(), and keys().
# There is no update() — corrections go through append_correction()
# (see patterns/03_correction_append.py).
store = InMemoryBitemporalStore()
store.append(record)
store.append(
    BitemporalRecord.new(
        key="ofac:cp-42",
        valid_time=datetime(2026, 4, 17, tzinfo=UTC),
        value={"sanctioned": False, "jurisdiction": "US"},
        source="ofac",
        source_trust_level="primary",
        transaction_time=datetime(2026, 4, 17, tzinfo=UTC),
    )
)

print(f"\nStore has {len(store)} records across keys: {sorted(store.keys())}")

# --- Read back ---
# latest() returns the most recent record for a key; history() returns all.
latest = store.latest("fx:USDC/USD")
print(f"\nstore.latest('fx:USDC/USD').value = {latest.value}")
print(f"len(store.history('fx:USDC/USD')) = {len(store.history('fx:USDC/USD'))}")

## 3. The correction pattern

When upstream restates a value, you don't mutate — you append a new record with the same `valid_time` and a fresh `transaction_time`. The correction links back via `parent_record_id`.

In [ ]:
from datetime import datetime, timedelta, timezone

from briefcase.bitemporal import (
    BitemporalRecord,
    InMemoryBitemporalStore,
    append_correction,
)

UTC = timezone.utc

# --- Original observation ---
# Bloomberg's original print lands on day 16: px=1.0001.
store = InMemoryBitemporalStore()
day_16 = datetime(2026, 4, 17, tzinfo=UTC)
original = BitemporalRecord.new(
    key="fx:USDC/USD",
    valid_time=day_16,
    value={"px": 1.0001, "size": 1_000_000},
    source="bloomberg",
    source_trust_level="primary",
    transaction_time=day_16,
)
store.append(original)
print(f"Original record: px={original.value['px']} at t={original.transaction_time.date()}")

# --- Correction appended ---
# 30 days later, Bloomberg issues a correction — the print should have been
# 1.0002. append_correction() creates a NEW record with the same valid_time
# and a new transaction_time; it links back via parent_record_id.
day_46 = day_16 + timedelta(days=30)
correction = append_correction(
    store,
    original,
    corrected_value={"px": 1.0002, "size": 1_000_000},
    transaction_time=day_46,
)
print(f"Correction:      px={correction.value['px']} at t={correction.transaction_time.date()}")
print(f"Correction.parent_record_id = {correction.parent_record_id}")
print(f"  (points back to original.record_id = {original.record_id})")

# --- Both records coexist ---
# The store holds both. `latest()` returns the correction (most recent
# transaction_time); `history()` returns both. The original belief is
# preserved and recoverable via AsOfView (see pattern 04).
hist = store.history("fx:USDC/USD")
print(f"\nStore history length: {len(hist)} records")
for r in hist:
    note = "(original)" if r.parent_record_id is None else "(correction)"
    print(f"  t={r.transaction_time.date()}  px={r.value['px']}  {note}")
print(f"\nstore.latest('fx:USDC/USD').value = {store.latest('fx:USDC/USD').value}")

## 4. Temporal replay — `AsOfView`

The payoff of patterns 02 and 03: wrap any bitemporal store in `AsOfView(store, t)` and reads clamp to what was known at time `t`. Production code and replay code share the same body.

In [ ]:
from datetime import datetime, timedelta, timezone

from briefcase.bitemporal import (
    AsOfView,
    BitemporalRecord,
    InMemoryBitemporalStore,
    append_correction,
)

UTC = timezone.utc

# --- Set up a store with a correction ---
# Original on day 16; correction on day 46. See pattern 03.
store = InMemoryBitemporalStore()
day_16 = datetime(2026, 4, 17, tzinfo=UTC)
original = BitemporalRecord.new(
    key="fx:USDC/USD",
    valid_time=day_16,
    value={"px": 1.0001, "size": 1_000_000},
    source="bloomberg",
    source_trust_level="primary",
    transaction_time=day_16,
)
store.append(original)
append_correction(
    store, original,
    corrected_value={"px": 1.0002, "size": 1_000_000},
    transaction_time=day_16 + timedelta(days=30),
)

# --- Three reads, same code, different clamps ---
# Live (no clamp), as-of day 30 (before correction), as-of day 50 (after).
print(f"Live view (no clamp):             {store.latest('fx:USDC/USD').value}")
with AsOfView(store, transaction_time=day_16 + timedelta(days=14)) as view:
    print(f"As-of day 30 (pre-correction):    {view.latest('fx:USDC/USD').value}")
with AsOfView(store, transaction_time=day_16 + timedelta(days=34)) as view:
    print(f"As-of day 50 (post-correction):   {view.latest('fx:USDC/USD').value}")

# --- Writes are refused on a view of the past ---
# An AsOfView is strictly read-only. Attempting to append through it
# raises, because a write into a historical view would leak post-as-of
# knowledge into the view's timeline.
view = AsOfView(store, transaction_time=day_16 + timedelta(days=14))
try:
    view.append(original)
except Exception as e:
    print(f"\nWrite refused on AsOfView: {type(e).__name__}: {e}")

## 5. Versioned policy

`PolicyRegistry` stores policy versions bitemporally. An auditor asking *'which rules were live on day X'* gets a reproducible answer — the v1 rule set is preserved after v2 is published.

In [ ]:
from datetime import datetime, timedelta, timezone

from briefcase.routing import PolicyRegistry, PolicyRule, PolicyVersion

UTC = timezone.utc


def _policy(version: str, description: str, rules: list[PolicyRule]) -> PolicyVersion:
    return PolicyVersion(
        policy_id="stablecoin_router",
        version=version,
        description=description,
        rules=rules,
        default_choice="human_review",
    )

# --- Publish v1 and v2 into the registry ---
# v1 live from day 0; v2 live from day 60. Publication is bitemporal —
# publishing v2 does not delete v1.
registry = PolicyRegistry()
day_0 = datetime(2026, 4, 1, tzinfo=UTC)

v1 = _policy(
    "1.0.0",
    "LATAM routes USDT for liquidity.",
    [PolicyRule(rule_id="latam_usdt", condition={"jurisdiction": "LATAM"}, choice="USDT", rationale="EM liquidity")],
)
v2 = _policy(
    "2.0.0",
    "LATAM routes USDC as compliant issuer distribution expands.",
    [PolicyRule(rule_id="latam_usdc", condition={"jurisdiction": "LATAM"}, choice="USDC", rationale="compliant issuer")],
)

registry.publish(v1, valid_from=day_0, transaction_time=day_0)
registry.publish(v2, valid_from=day_0 + timedelta(days=60), transaction_time=day_0 + timedelta(days=60))

print("Registry history for 'stablecoin_router':")
for p in registry.history("stablecoin_router"):
    print(f"  {p.version}: {p.description}")

# --- As-of reads pick the right version ---
# Reading "as-of day 50" returns v1; "as-of day 90" returns v2.
for days in (50, 90):
    p = registry.get("stablecoin_router", as_of_transaction_time=day_0 + timedelta(days=days))
    print(f"\nAs-of day {days}: policy version = {p.version if p else '(none)'}")

## 6. The examiner bundle

`ExaminerBundle.build()` joins a decision + the policy-as-of decision + the evidence records into a single content-addressed JSON artifact. `verify()` detects any tamper.

In [ ]:
import json
from datetime import datetime, timedelta, timezone

from briefcase.bitemporal import BitemporalRecord, InMemoryBitemporalStore
from briefcase.compliance import BundleIntegrityError, ExaminerBundle
from briefcase.routing import (
    AgentRoutingDecision,
    PolicyRegistry,
    PolicyRule,
    PolicyVersion,
)

UTC = timezone.utc

# --- Evidence + policy registry ---
# One evidence record plus a minimal policy. In a real system both come
# from the existing bitemporal store + registry; the bundle just joins them.
day_0 = datetime(2026, 4, 1, tzinfo=UTC)
evidence = InMemoryBitemporalStore()
fx = BitemporalRecord.new(
    key="fx:USDC/USD",
    valid_time=day_0,
    value={"px": 1.0001},
    source="bloomberg",
    source_trust_level="primary",
    transaction_time=day_0,
)
evidence.append(fx)

registry = PolicyRegistry()
registry.publish(
    PolicyVersion(
        policy_id="fx_check",
        version="1.0.0",
        description="Block if FX px deviates from peg by >2bps.",
        rules=[PolicyRule(rule_id="peg_check", condition={"within_peg": True}, choice="allow", rationale="within peg")],
        default_choice="block",
    ),
    valid_from=day_0,
    transaction_time=day_0,
)

# --- Build and fingerprint the bundle ---
# A decision (projected here for demo) references the evidence by record_id.
# ExaminerBundle.build() clamps policy to decided_at and captures everything.
decision = AgentRoutingDecision(
    decision_id="demo-001",
    use_case="fx_routing",
    context={"px": 1.0001},
    candidates=["allow", "block"],
    selected="allow",
    policy_id="fx_check",
    policy_version="1.0.0",
    matched_rule_id="peg_check",
    evidence_refs=[fx.record_id],
    rationale="within peg",
    decided_at=day_0 + timedelta(hours=1),
)

bundle = ExaminerBundle.build(
    decision,
    evidence_store=evidence,
    policy_registry=registry,
    metadata={"doc": "pattern 06 demo"},
)
print(f"Bundle assembled:")
print(f"  content_hash:   {bundle.content_hash}")
print(f"  policy version: v{bundle.policy['version']}")
print(f"  evidence rows:  {len(bundle.evidence)}")

# --- Verify, round-trip, tamper-check ---
# verify() passes on the untouched bundle and after JSON round-trip; any
# mutation (here: flipping the decision) causes it to raise.
bundle.verify()
print(f"\nverify() on untouched bundle:    OK")

payload = bundle.to_json(indent=2)
ExaminerBundle.from_json(payload).verify()
print(f"verify() after JSON round-trip:  OK")

tampered = json.loads(payload)
tampered["decision"]["selected"] = "block"
try:
    ExaminerBundle.from_dict(tampered).verify()
except BundleIntegrityError as e:
    print(f"verify() on tampered bundle:     REJECTED ({type(e).__name__})")

## 7. Cost attribution

`CostCalculator` turns model + token usage into deterministic per-decision cost using real provider pricing.

In [ ]:
from briefcase.cost import CostCalculator

calc = CostCalculator()

# --- Per-decision cost for one model ---
# estimate_cost takes model + input/output tokens and returns the cost in
# USD using real provider pricing compiled into the SDK.
c = calc.estimate_cost("gpt-4o", input_tokens=1200, output_tokens=350)
print(f"gpt-4o 1200 in / 350 out: ${c.total_cost:.6f}")
print(f"  input portion:  ${c.input_cost:.6f}")
print(f"  output portion: ${c.output_cost:.6f}")

# --- Cross-model comparison ---
# estimate_cost returns a CostEstimate with total_cost, input_cost, etc.
# Running it for multiple models supports routing decisions such as
# "cheapest model that meets the quality bar."
for model in ["gpt-4o", "claude-3-5-sonnet", "gpt-4o-mini"]:
    try:
        est = calc.estimate_cost(model, 1200, 350)
        print(f"  {model:<25} ${est.total_cost:.6f}")
    except Exception as e:
        print(f"  {model:<25} (not registered: {type(e).__name__})")

# --- Monthly projection ---
# Scale a single-decision cost by expected traffic. Use this for budget
# planning and proactive alerts before month-end overruns.
per_day = 50_000
per_call = calc.estimate_cost("gpt-4o-mini", 800, 200)
monthly = per_call.total_cost * per_day * 30
print(f"\nProjection for 50k gpt-4o-mini calls/day: ${monthly:,.2f}/month")

## 8. Drift detection

`DriftCalculator` measures how far a recent window of outputs has moved from a baseline.

In [ ]:
from briefcase.drift import DriftCalculator

# --- Baseline vs. current distribution ---
# Feed the calculator a batch of outputs (strings); it computes the internal
# similarity structure and reports a drift score. Thresholds around 0.7-0.8
# are reasonable defaults for "meaningfully different."
baseline = [
    "approved: credit score 720, debt-to-income 0.28",
    "approved: credit score 705, debt-to-income 0.31",
    "approved: credit score 740, debt-to-income 0.22",
    "denied: credit score 610, debt-to-income 0.44",
]
current = [
    "approved: credit score 680, debt-to-income 0.35",
    "approved: credit score 690, debt-to-income 0.33",
    "denied: credit score 600, debt-to-income 0.45",
    "denied: credit score 625, debt-to-income 0.42",
]

calc = DriftCalculator()
baseline_m = calc.calculate_drift(baseline)
current_m = calc.calculate_drift(current)
print(f"Baseline: drift={baseline_m.drift_score:.4f}  consistency={baseline_m.consistency_score:.4f}")
print(f"Current:  drift={current_m.drift_score:.4f}  consistency={current_m.consistency_score:.4f}")

# --- Combined pool ---
# calculate_drift on baseline+current together surfaces whether the two
# windows cluster apart — the classic "something changed" signal.
combined = baseline + current
combined_m = calc.calculate_drift(combined)
print(f"Combined: drift={combined_m.drift_score:.4f}  samples={combined_m.total_samples}")
print(
    "\nInterpretation: combined drift notably higher than either window\n"
    "alone indicates the two populations are pulling apart. In production\n"
    "this is the automated signal to page a model owner."
)

## 9. PII sanitization

The `Sanitizer` detects and redacts PII in free text and structured payloads.

In [ ]:
import json

from briefcase.sanitize import Sanitizer

sanitizer = Sanitizer()

# --- Detect and redact PII in free text ---
# sanitize() returns the scrubbed text; contains_pii() is a quick bool
# check useful for gating (e.g., "never send PII to this vendor").
dirty = (
    "Subject: Mr. John Smith. SSN: 123-45-6789. "
    "Reachable at john.smith@example.com or (415) 555-0199. "
    "Card on file: 4111 1111 1111 1111."
)
print("Before sanitization:")
print(f"  {dirty}")
print(f"\nSanitizer.contains_pii(): {sanitizer.contains_pii(dirty)}")

result = sanitizer.sanitize(dirty)
print(f"\nAfter sanitization:")
print(f"  {result.sanitized}")
print(f"  redactions: {result.redaction_count}")

# --- Inspect the match set ---
# analyze_pii() returns a dict describing what matched, so you can log
# counts / types without logging the underlying values. Useful for
# observability on redaction rate.
analysis = sanitizer.analyze_pii(dirty)
print(f"\nanalyze_pii() summary:")
print(f"  has_pii:         {analysis['has_pii']}")
print(f"  total_matches:   {analysis['total_matches']}")
print(f"  detected_types:  {analysis['detected_types']}")

# --- Sanitize structured payloads ---
# sanitize_json() walks a JSON-serializable payload and redacts PII in
# any string value. Keys are preserved.
record = {
    "id": "case-2026-04-01",
    "summary": "Witness interviewed at home (phone 415-555-0100).",
    "officer_email": "officer.smith@policedept.gov",
}
print(f"\nOriginal record:  {json.dumps(record)}")
json_res = sanitizer.sanitize_json(json.dumps(record))
print(f"Sanitized record: {json_res.sanitized if hasattr(json_res, 'sanitized') else json_res}")

## 10. Guardrail pipeline

`GuardrailPipeline` composes multiple `GuardrailEnv` stages into a single allow/deny check. `FIRST_DENY` mode short-circuits on the first stage that says no.

In [ ]:
from briefcase.guardrails import (
    BaseGuardrailEnv,
    Effect,
    EnvSpec,
    EvalRequest,
    EvalResult,
    GuardrailPipeline,
    PolicySpace,
)


class MaxNotionalEnv(BaseGuardrailEnv):
    """Deny requests whose context.notional_usd exceeds a hard cap."""

    def __init__(self, cap: float = 100_000.0):
        self._cap = cap
        self._spec = EnvSpec(id="max_notional", entry_point=f"{__name__}:MaxNotionalEnv")

    @property
    def name(self) -> str:
        return "max_notional"

    @property
    def request_space(self) -> PolicySpace:
        return PolicySpace(dimensions={}, constraints=[])

    def evaluate(self, request: EvalRequest) -> EvalResult:
        notional = request.context.get("notional_usd", 0)
        if notional > self._cap:
            return EvalResult(effect=Effect.DENY, guardrail_name=self.name, reason=f"notional {notional} exceeds cap {self._cap}")
        return EvalResult(effect=Effect.ALLOW, guardrail_name=self.name, reason="under cap")

    def explain(self, request: EvalRequest, result: EvalResult):
        return result

    def reset(self) -> None:
        pass

    def close(self) -> None:
        pass


class JurisdictionAllowlistEnv(BaseGuardrailEnv):
    """Allow only a configured set of jurisdictions."""

    def __init__(self, allowed=("US", "EU", "UK")):
        self._allowed = set(allowed)
        self._spec = EnvSpec(id="juris_allow", entry_point=f"{__name__}:JurisdictionAllowlistEnv")

    @property
    def name(self) -> str:
        return "juris_allow"

    @property
    def request_space(self) -> PolicySpace:
        return PolicySpace(dimensions={}, constraints=[])

    def evaluate(self, request: EvalRequest) -> EvalResult:
        j = request.context.get("jurisdiction", "??")
        if j not in self._allowed:
            return EvalResult(effect=Effect.DENY, guardrail_name=self.name, reason=f"jurisdiction {j} not in {sorted(self._allowed)}")
        return EvalResult(effect=Effect.ALLOW, guardrail_name=self.name, reason=f"{j} allowed")

    def explain(self, request: EvalRequest, result: EvalResult):
        return result

    def reset(self) -> None:
        pass

    def close(self) -> None:
        pass

# --- Compose two guardrails into a pipeline ---
# FIRST_DENY mode short-circuits: if any stage denies, the pipeline denies
# with that stage's reason. Stages run in order.
pipeline = GuardrailPipeline(
    stages=[JurisdictionAllowlistEnv(), MaxNotionalEnv(cap=100_000)],
    name="payment_checks",
)

# --- Exercise three cases ---
cases = [
    ("allowed", EvalRequest(agent="a1", action="route_payment", resource="corridor_us_eu",
                            context={"jurisdiction": "US", "notional_usd": 25_000})),
    ("bad_jurisdiction", EvalRequest(agent="a1", action="route_payment", resource="corridor_kp",
                            context={"jurisdiction": "KP", "notional_usd": 25_000})),
    ("over_cap", EvalRequest(agent="a1", action="route_payment", resource="corridor_us_eu",
                            context={"jurisdiction": "US", "notional_usd": 500_000})),
]
for label, req in cases:
    result = pipeline.evaluate(req)
    first = result.individual_results[-1]  # the deciding stage
    print(f"  [{label:<18}] final={result.final_effect.value:<5} short_circuited={result.short_circuited:<5} reason={first.reason}")

print(
    "\nThe FIRST_DENY short-circuit is what makes guardrails fast: for the\n"
    "over_cap request, the jurisdiction stage allows then the cap stage\n"
    "denies — but a request with a bad jurisdiction short-circuits on\n"
    "stage 1 and never runs the cap check."
)

## 11. Decision replay

Captured snapshots store the inputs that produced each output. The replay pattern re-runs a candidate against those stored inputs and diffs — the regression-test primitive.

In [ ]:
import briefcase
from briefcase import DecisionSnapshot, Input, ModelParameters, Output
from briefcase.storage import SqliteBackend

briefcase.init()


def classify_v1(document: str) -> dict:
    """Original classifier."""
    score = 0.92 if "great" in document.lower() else 0.31
    return {"label": "positive" if score > 0.5 else "negative", "score": score}


def classify_v2(document: str) -> dict:
    """Candidate replacement — wider positive vocabulary."""
    positive_terms = ("great", "excellent", "love", "amazing")
    score = 0.95 if any(t in document.lower() for t in positive_terms) else 0.28
    return {"label": "positive" if score > 0.5 else "negative", "score": score}


backend = SqliteBackend.in_memory()

# --- Capture baseline decisions into storage ---
# Build DecisionSnapshots for each v1 call and save to the backend.
# In production this is handled by @capture (see pattern 01); here we
# construct them explicitly so the replay step is self-contained.
docs = ["The product is great", "This is terrible", "Excellent value"]
snapshot_ids: list[str] = []
for doc in docs:
    out = classify_v1(doc)
    snap = DecisionSnapshot("classify_v1")
    snap.add_input(Input("document", doc, "string"))
    snap.with_model_parameters(ModelParameters("classify_v1"))
    snap.add_output(Output("label", out["label"], "string").with_confidence(out["score"]))
    snapshot_ids.append(backend.save_decision(snap))
print(f"Captured {len(snapshot_ids)} baseline decisions")

# --- Replay against the candidate ---
# Load each snapshot, re-run the candidate with the stored input, and
# compare outputs. The match rate is the regression-test signal.
matches, diffs = 0, []
for sid in snapshot_ids:
    original = backend.load_decision(sid)
    doc = next(i.value for i in original.inputs if i.name == "document")
    original_label = next(o.value for o in original.outputs if o.name == "label")
    new_out = classify_v2(doc)
    if new_out["label"] == original_label:
        matches += 1
    else:
        diffs.append((doc, original_label, new_out["label"]))

print(f"\nReplay against classify_v2: {matches}/{len(snapshot_ids)} matched on label")
for doc, orig, new in diffs:
    print(f"  DIFF: {doc!r} -- v1={orig} v2={new}")

# --- Why this is the regression-test primitive ---
print(
    "\nThe replay loop is the regression test: load historical inputs,\n"
    "re-run the candidate, diff outputs. A match rate of 100% means the\n"
    "change is safe for backfill; anything less flags what moved and how."
)

## Where to go next

- **Apply these in a domain:** see [`agentic-payments/`](../agentic-payments/) for patterns 02–06 composed into a cross-border payments story, or [`regulatory-workflows/02_ofac_sanctions/`](../regulatory-workflows/02_ofac_sanctions/) for those same primitives applied to sanctions screening.
- **Build your own composition:** open the per-pattern `.py` files and copy the pieces you need. Each is <60 lines and self-contained.
- **Trace an example back to primitives:** the [composition matrix](README.md#composition-matrix) maps each domain example back to the patterns it uses.